# 03 양서연 · TOPIS 통행속도와 날씨 결합
민주의 TOPIS 월별 CSV 40개와 정희의 시간별 날씨를 결합해 도로×방향×출퇴근 시간 통행시간 표를 만든다. 원본은 수정하지 않는다.

규칙: 연구기간의 평일, 출근 `~08·~09시`, 퇴근 `~18·~19·~20시`, 도시고속도로·주간선도로만 사용한다. `~08시`는 07:00~08:00 평균이므로 날씨 `hour=8`과 결합한다.

## 1. 환경·경로 확인 — 파일 저장 없음

In [ ]:
import importlib.util, os, sys
from pathlib import Path

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'README.md').is_file() and (candidate / '.git').exists():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')

PROJECT_ROOT = find_project_root()
DATA_ROOT = Path(os.getenv('SEOUL_TRAFFIC_DATA_ROOT', r'D:\seoul-weather-traffic-data'))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print('Python:', sys.executable)
print('프로젝트:', PROJECT_ROOT)
print('데이터:', DATA_ROOT)


## 2. 패키지·입력 확인 — 파일 저장 없음
`pyarrow`가 없으면 자동 설치하지 않고 중단한다.

In [ ]:
required = ['pandas', 'pyarrow']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f'필요 패키지가 없습니다: {missing}')
topis_paths = sorted((DATA_ROOT / 'topis_speed' / 'csv').glob('topis_speed_*.csv'))
weather_paths = sorted((DATA_ROOT / 'weather' / 'processed').rglob('weather_hourly_*.csv'))
print('TOPIS 월 파일:', len(topis_paths))
print('시간별 날씨 파일:', len(weather_paths))
if len(topis_paths) != 40 or len(weather_paths) != 2:
    raise RuntimeError('입력 파일 수를 확인하세요.')


## 3. 결합 함수 불러오기 — 파일 저장 없음

In [ ]:
from src.yang_seoyeon_merge import (
    COMMUTE_HOURS, STUDY_PERIODS, TARGET_FUNCTION_TYPES, run_merge,
)
print('출퇴근 종료시각:', COMMUTE_HOURS)
print('기능유형:', TARGET_FUNCTION_TYPES)
print('연구기간 수:', len(STUDY_PERIODS))


## 4. 전체 결합 실행 — **대용량 읽기 및 파일 저장 발생**
이 셀에서만 TOPIS 40개월을 실제 처리한다. 전체 Parquet은 D드라이브에 저장하고, 교수님 확인용 월별 요약·200행 샘플·실행 로그는 프로젝트 폴더에 저장한다. 기존 파일은 덮어쓰지 않고 다음 버전을 사용한다.

In [ ]:
merge_log = run_merge(DATA_ROOT, PROJECT_ROOT)
for key in ['output_rows', 'date_min', 'date_max', 'unique_roads', 'duplicate_keys', 'missing_weather_rows', 'all_speed_missing_rows', 'output_path', 'summary_path', 'sample_path']:
    print(f'{key}: {merge_log[key]}')


## 5. 해석 주의
시기별 도로망 변경 때문에 월별 대상 도로 수가 89~91개일 수 있다. 도로명을 임의로 통합하지 않고 원자료의 실제 명칭을 보존한다. 전체 속도가 결측인 도로·시간은 통행시간도 NaN이며 0으로 바꾸지 않는다. 시간 열은 구간 종료시각 기준이다.